In [1]:
import pandas as pd
import re
from difflib import SequenceMatcher

In [2]:
REQUIRED_COLUMNS = {"Date", "Name", "Year", "Letterboxd URI", "Rating"}

In [3]:
def load_letterboxd_export(csv_path: str) -> pd.DataFrame:
    """
    Load a user's Letterboxd ratings export.
    Export from: Letterboxd Settings -> Import & Export -> Export Your Data
    (ratings.csv inside the downloaded zip)
    """
    df = pd.read_csv(csv_path)

    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"Unexpected Letterboxd export format, missing columns: {missing}")

    df = df.rename(columns={
        "Name": "title",
        "Year": "year",
        "Letterboxd URI": "letterboxd_url",
        "Rating": "rating",
        "Date": "date_rated",
    })

    df["rating"] = df["rating"].astype(float)
    df["year"] = df["year"].astype("Int64")  # nullable int, Letterboxd sometimes leaves this blank
    return df[["title", "year", "rating", "letterboxd_url", "date_rated"]]

In [6]:
def normalize_title(title: str) -> str:
    """Lowercase, strip punctuation/articles for fuzzier matching."""
    title = title.lower().strip()
    title = re.sub(r"^(the|a|an)\s+", "", title)
    title = re.sub(r"[^\w\s]", "", title)
    return title.strip()

def title_similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_title(a), normalize_title(b)).ratio()

In [7]:
def map_to_movielens(
    user_ratings: pd.DataFrame,
    lookup_path: str = "models/movie_lookup.pkl",
    similarity_threshold: float = 0.90,
) -> pd.DataFrame:
    """
    Match Letterboxd (title, year) pairs to MovieLens movieIds using
    movie_lookup.pkl (indexed by movieId, has a 'title' column).

    Strategy:
      1. Exact match on (normalized title, year) -- fast, catches ~90%+
      2. Fuzzy fallback on title within same year for anything unmatched
    """
    lookup = pd.read_pickle(lookup_path)

    lookup_movies = (
        lookup.reset_index()[["movieId", "title"]]
        .drop_duplicates(subset="movieId")
        .copy()
    )

    # MovieLens titles are formatted like "Toy Story (1995)" -- pull year out
    year_extract = lookup_movies["title"].str.extract(r"\((\d{4})\)$")
    lookup_movies["year"] = pd.to_numeric(year_extract[0], errors="coerce").astype("Int64")
    lookup_movies["title_clean"] = (
        lookup_movies["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True)
    )
    lookup_movies["title_norm"] = lookup_movies["title_clean"].apply(normalize_title)

    user_ratings = user_ratings.copy()
    user_ratings["title_norm"] = user_ratings["title"].apply(normalize_title)

    # Pass 1: exact match on (title_norm, year)
    exact = user_ratings.merge(
        lookup_movies[["movieId", "title_norm", "year"]],
        on=["title_norm", "year"],
        how="left",
    )

    matched_mask = exact["movieId"].notna()
    matched = exact[matched_mask].copy()
    unmatched = exact[~matched_mask].drop(columns=["movieId"])

    # Pass 2: fuzzy match within same year for whatever pass 1 missed
    fuzzy_matches = []
    if len(unmatched) > 0:
        for _, row in unmatched.iterrows():
            candidates = lookup_movies[lookup_movies["year"] == row["year"]]
            if candidates.empty:
                continue

            scores = candidates["title_clean"].apply(lambda t: title_similarity(t, row["title"]))
            best_idx = scores.idxmax()
            best_score = scores.loc[best_idx]

            if best_score >= similarity_threshold:
                match_row = row.to_dict()
                match_row["movieId"] = candidates.loc[best_idx, "movieId"]
                fuzzy_matches.append(match_row)

    if fuzzy_matches:
        fuzzy_df = pd.DataFrame(fuzzy_matches)
        matched = pd.concat([matched, fuzzy_df], ignore_index=True)
        still_unmatched = unmatched[~unmatched["title"].isin(fuzzy_df["title"])]
    else:
        still_unmatched = unmatched

    if len(still_unmatched) > 0:
        print(f"Warning: {len(still_unmatched)}/{len(exact)} titles had no MovieLens match.")
        print(still_unmatched[["title", "year"]].to_string(index=False))

    matched["movieId"] = matched["movieId"].astype(int)
    return matched

In [11]:
def build_new_user_profile(
    csv_path: str,
    lookup_path: str = "models/movie_lookup.pkl",
) -> pd.DataFrame:
    """Full pipeline: Letterboxd export -> MovieLens-compatible ratings dataframe."""
    raw = load_letterboxd_export(csv_path)
    mapped = map_to_movielens(raw, lookup_path)
    # No rating rescale needed -- Letterboxd is already 0.5-5.0
    return mapped[["movieId", "rating"]]